# 02 — Apriori Basket Analysis (Co-Purchase Edges)

**Goal:** find frequent co-purchase pairs from sessionised baskets and export them as a weighted edge list for the community-detection notebook.

**Inputs:** `data/interim/sample_30d.parquet` (from 01)
**Outputs:** `data/interim/cooccurrence_edges.parquet`

**Method:** mlxtend Apriori on a transactional encoding of (customer, day) → set of articles. Support threshold 0.001, lift > 1.5. The numbers are documented and defensible in `data/methodology.md`.

**What this notebook is NOT:** a recommender. We're surfacing the *structure* of co-purchase, not predicting future purchases. See methodology.md §"What the H&M co-purchase data can and cannot conclude."

## 1 · Setup + load sample

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
from tqdm import tqdm

INTERIM = Path('../interim')

tx = pd.read_parquet(INTERIM / 'sample_30d.parquet')
articles = pd.read_parquet(INTERIM / 'articles_clean.parquet')

print(f"Loaded {len(tx):,} transactions, {tx['article_id'].nunique():,} unique articles")

Loaded 1,155,933 transactions, 29,237 unique articles


## 2 · Sessionise into baskets

Same definition as in notebook 01: one (customer_id, t_dat) = one basket. We immediately drop singleton baskets — they contribute no co-purchase signal.

In [2]:
baskets = (
    tx.groupby(['customer_id', 't_dat'])['article_id']
    .apply(lambda s: list(set(s)))  # dedupe within basket
    .reset_index(name='items')
)
baskets['size'] = baskets['items'].str.len()
baskets = baskets[baskets['size'] >= 2].reset_index(drop=True)
print(f"Usable multi-item baskets: {len(baskets):,}")
print(f"Total item-slots in those baskets: {baskets['size'].sum():,}")
baskets.head(3)

Usable multi-item baskets: 237,091
Total item-slots in those baskets: 917,186


,customer_id,t_dat,items,size
0,0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d...,2020-09-14,"[0719530003, 0448509014]",2
1,00009d946eec3ea54add5ba56d5210ea898def4b46c685...,2020-08-30,"[0516859008, 0797892001, 0889652001, 056224509...",6
2,0000b2f1829e23b24feec422ef13df3ccedaedc85368e6...,2020-08-29,"[0914441005, 0706016038, 0706016015, 0778476005]",4


## 3 · Cap rare items before Apriori

Apriori scales poorly with rare items — and items appearing in <50 baskets in our window can't produce statistically meaningful support anyway. We cap at the top N most-frequent articles to keep the matrix tractable. This is the standard practical compromise; documented in methodology.md.

In [3]:
TOP_N_ARTICLES = 5000  # tune based on your machine. 5000 keeps the matrix around 600MB.

item_counts = pd.Series(
    [a for items in baskets['items'] for a in items]
).value_counts()

kept = item_counts.head(TOP_N_ARTICLES).index
print(f"Kept top {TOP_N_ARTICLES:,} articles (out of {item_counts.shape[0]:,})")
print(f"Coverage: {item_counts.head(TOP_N_ARTICLES).sum() / item_counts.sum():.1%} of all item-slots")

kept_set = set(kept)
baskets['items_kept'] = baskets['items'].apply(lambda items: [a for a in items if a in kept_set])
baskets['size_kept'] = baskets['items_kept'].str.len()
baskets = baskets[baskets['size_kept'] >= 2].reset_index(drop=True)
print(f"Baskets remaining after capping: {len(baskets):,}")

Kept top 5,000 articles (out of 28,366)
Coverage: 80.8% of all item-slots


Baskets remaining after capping: 198,255


## 4 · Sparse co-occurrence (replaces Apriori at this scale)

**Why not Apriori:** at 198k baskets × 5k items, Apriori OOMs at low support (need <0.0001 to surface long-tail style pairs) and over-prunes at higher support (>=0.0002 produced only 19 edges, all mega-popular same-category bestseller pairs).

**What instead:** sparse one-hot matrix `X`, co-occurrence is just `X.T @ X` — a single sparse matmul that gives all pairwise counts in O(nnz). We derive lift from counts directly, then threshold by `min_cooccurrence_count` and `min_lift`. Faster, finer control, no combinatorial blowup. Methodologically equivalent for our purpose (pair-level edge list for graph analysis).

In [4]:
from scipy.sparse import csr_matrix
import numpy as np

# Build sparse one-hot directly from basket lists (no TransactionEncoder needed)
all_items = sorted({a for items in baskets['items_kept'] for a in items})
item_to_idx = {a: i for i, a in enumerate(all_items)}
n_baskets = len(baskets)
n_items = len(all_items)

rows, cols = [], []
for bidx, items in enumerate(baskets['items_kept']):
    for a in items:
        rows.append(bidx)
        cols.append(item_to_idx[a])

X = csr_matrix((np.ones(len(rows), dtype=np.int32), (rows, cols)), shape=(n_baskets, n_items))
print(f"Sparse one-hot: {X.shape}, nnz={X.nnz:,}")

# Pairwise co-occurrence: X.T @ X is item×item matrix of basket counts
cooc = (X.T @ X).tocsr()
item_freq = np.asarray(X.sum(axis=0)).flatten()
print(f"Co-occurrence matrix: {cooc.shape}, total nnz={cooc.nnz:,} (incl. diagonal)")

Sparse one-hot: (198255, 5000), nnz=712,725
Co-occurrence matrix: (5000, 5000), total nnz=1,830,110 (incl. diagonal)


In [5]:
# Filter pairs vectorised: upper triangle only, min co-occurrence count, min lift
MIN_COOC = 10    # pair appears in >=10 baskets — guarantees not just a chance pair of singletons
MIN_LIFT = 1.5   # co-occurs >=1.5x more than independent expectation

cooc_coo = cooc.tocoo()
upper = cooc_coo.row < cooc_coo.col
i = cooc_coo.row[upper]
j = cooc_coo.col[upper]
c = cooc_coo.data[upper]

# Threshold 1: min co-occurrence count
mask = c >= MIN_COOC
i, j, c = i[mask], j[mask], c[mask]

# Compute lift vectorised
lift = (c.astype(np.float64) * n_baskets) / (item_freq[i] * item_freq[j])

# Threshold 2: min lift
mask = lift >= MIN_LIFT
i, j, c, lift = i[mask], j[mask], c[mask], lift[mask]

items_arr = np.array(all_items)
edges = pd.DataFrame({
    'source': items_arr[i],
    'target': items_arr[j],
    'lift': lift,
    'support': c / n_baskets,
    'confidence': c / item_freq[i],
})
edges = edges.sort_values('lift', ascending=False).reset_index(drop=True)
print(f"Edges at min_cooc>={MIN_COOC}, min_lift>={MIN_LIFT}: {len(edges):,}")
print(f"Lift range: {edges['lift'].min():.2f} → {edges['lift'].max():.2f}, median {edges['lift'].median():.2f}")
edges.head(10)

Edges at min_cooc>=10, min_lift>=1.5: 8,603
Lift range: 1.50 → 4262.24, median 20.00


,source,target,lift,support,confidence
0,0850176003,0854619003,4262.238943,0.000177,0.795455
1,0854826001,0854830001,4019.481216,0.000171,0.871795
2,0799409001,0799410001,3960.440658,0.000171,0.739130
3,0833499003,0833530003,3552.161254,0.000161,0.842105
4,0811907007,0865558001,3521.131274,0.000116,0.657143
5,0629746002,0629758005,3460.532095,0.000156,0.837838
6,0889974002,0914414002,3355.379594,0.000177,0.795455
7,0882757001,0882759001,3352.137681,0.000177,0.777778
8,0713995001,0714032001,3346.906653,0.000171,0.894737
9,0802979001,0802980001,3218.425325,0.000177,0.795455


In [6]:
# `edges` is already built as the undirected weighted edge list — no association_rules step needed.
# Sanity-print and continue.
print(f"Final edges DataFrame: {len(edges):,} rows, columns = {list(edges.columns)}")
print(f"\nUnique nodes in edge list: {len(set(edges['source']).union(set(edges['target']))):,}")

Final edges DataFrame: 8,603 rows, columns = ['source', 'target', 'lift', 'support', 'confidence']

Unique nodes in edge list: 2,764


## 5 · Edge list ready (skipped Apriori section)

`edges` is the undirected weighted edge list with `source`, `target`, `lift`, `support`, `confidence`. The next cell is a no-op kept only to preserve cell-numbering for downstream notebooks; jump to section 6.

In [7]:
# No-op: edges already built directly in cell 9.
pass

## 6 · Sanity check: do edges cross garment groups?

If the case study's claim — *style-coherent recommendations naturally cross garment groups* — holds in the data, then high-lift edges should disproportionately span different garment_group_name values. We check that here, and the number goes into the case-study narrative.

In [8]:
art_lookup = articles.set_index('article_id')['garment_group_name'].to_dict()
edges['source_group'] = edges['source'].map(art_lookup)
edges['target_group'] = edges['target'].map(art_lookup)
edges['cross_group'] = edges['source_group'] != edges['target_group']

print(f"Cross-garment-group edges: {edges['cross_group'].sum():,} ({edges['cross_group'].mean():.1%})")
print("\nAmong top 1000 highest-lift edges:")
top = edges.sort_values('lift', ascending=False).head(1000)
print(f"  Cross-group share: {top['cross_group'].mean():.1%}")
# This 'cross-group share' is the headline number for the case study's data section.
# If high (e.g., >50%), it strongly supports the cross-category-coverage principle.

Cross-garment-group edges: 2,294 (26.7%)

Among top 1000 highest-lift edges:
  Cross-group share: 8.6%


In [9]:
# Quick look at top cross-group edges — these are the most case-study-interesting
top_cross = edges[edges['cross_group']].sort_values('lift', ascending=False).head(10).copy()
top_cross['source_type'] = top_cross['source'].map(articles.set_index('article_id')['product_type_name'].to_dict())
top_cross['target_type'] = top_cross['target'].map(articles.set_index('article_id')['product_type_name'].to_dict())
top_cross[['source_type', 'source_group', 'target_type', 'target_group', 'lift']]

,source_type,source_group,target_type,target_group,lift
8,Blazer,Dressed,Trousers,Trousers,3346.906653
55,Trousers,Trousers,Blazer,Dressed,1488.400901
63,Trousers,Trousers,Blazer,Dressed,1359.462857
73,Underwear Tights,Socks and Tights,Underwear bottom,"Under-, Nightwear",1313.737952
77,Jacket,Blouses,Skirt,Trousers,1252.325162
91,Blazer,Dressed,Trousers,Trousers,1113.050243
127,Trousers,Trousers,Blazer,Outdoor,899.197206
237,Socks,"Under-, Nightwear",Hat/beanie,Accessories,598.596014
262,Skirt,Skirts,Blazer,Dressed,558.978241
275,Trousers,Trousers,Blazer,Dressed,533.364122


## 7 · Anchor check — light-blue denim co-purchases

Reference category from notebook 01. We expect the anchor SKUs to appear in the edge list co-purchased with tops, shoes, bags — the case study's specific claim about the redesigned Complete the Look widget. If the anchor SKUs appear ONLY with other denim, that's actually a *finding worth reporting* — it would reinforce that current widgets stay same-category.

In [10]:
anchor_skus = set(pd.read_parquet(INTERIM / 'anchor_skus.parquet')['article_id'])
anchor_edges = edges[
    edges['source'].isin(anchor_skus) | edges['target'].isin(anchor_skus)
]
print(f"Edges touching anchor SKUs: {len(anchor_edges):,}")
anchor_edges['source_type'] = anchor_edges['source'].map(articles.set_index('article_id')['product_type_name'].to_dict())
anchor_edges['target_type'] = anchor_edges['target'].map(articles.set_index('article_id')['product_type_name'].to_dict())
print("\nProduct types co-purchased with anchor SKUs:")
pd.concat([anchor_edges['source_type'], anchor_edges['target_type']]).value_counts().head(10)

Edges touching anchor SKUs: 63



Product types co-purchased with anchor SKUs:


Trousers    121
Shorts        2
Shirt         2
Unknown       1
Name: count, dtype: int64

## 8 · Write edges for notebook 03

In [11]:
edges.to_parquet(INTERIM / 'cooccurrence_edges.parquet', index=False, compression='snappy')
sz = (INTERIM / 'cooccurrence_edges.parquet').stat().st_size / 1024 / 1024
print(f"Wrote cooccurrence_edges.parquet ({sz:.1f} MB, {len(edges):,} edges)")
print("\nNext: open notebook 03-community-detection.ipynb")

Wrote cooccurrence_edges.parquet (0.2 MB, 8,603 edges)

Next: open notebook 03-community-detection.ipynb
